In [7]:
import os

dataset_path = "../data/dice_files/"

for root, dirs, files in os.walk(dataset_path):
    print(f"\nFolder: {root}")
    print("Subfolders:", len(dirs))
    print("Files:", len(files))

    for f in files[:5]:
        print(" ", f)


Folder: ../data/dice_files/
Subfolders: 2
Files: 0

Folder: ../data/dice_files/1.3.6.1.4.1.55648.52489457771155006307501275967031550463
Subfolders: 8
Files: 0

Folder: ../data/dice_files/1.3.6.1.4.1.55648.52489457771155006307501275967031550463/1.3.6.1.4.1.55648.52489457771155006307501275967031550463.2
Subfolders: 0
Files: 1
  1.3.6.1.4.1.55648.52489457771155006307501275967031550463.2.2.green.dcm

Folder: ../data/dice_files/1.3.6.1.4.1.55648.52489457771155006307501275967031550463/1.3.6.1.4.1.55648.52489457771155006307501275967031550463.5
Subfolders: 0
Files: 26
  1.3.6.1.4.1.55648.52489457771155006307501275967031550463.5.19.green.dcm
  1.3.6.1.4.1.55648.52489457771155006307501275967031550463.5.3.green.dcm
  1.3.6.1.4.1.55648.52489457771155006307501275967031550463.5.14.green.dcm
  1.3.6.1.4.1.55648.52489457771155006307501275967031550463.5.21.green.dcm
  1.3.6.1.4.1.55648.52489457771155006307501275967031550463.5.5.green.dcm

Folder: ../data/dice_files/1.3.6.1.4.1.55648.524894577711550063

In [8]:
import pydicom
import os

# Get path to first file in first series
dataset_path = "../data/dice_files/"
first_series = None
first_file   = None

for root, dirs, files in os.walk(dataset_path):
    dcm_files = [f for f in files if f.endswith('.dcm')]
    if dcm_files:
        first_series = root
        first_file   = os.path.join(root, dcm_files[0])
        break

print(f"Reading: {first_file}\n")
dcm = pydicom.dcmread(first_file)

# Print full metadata
print("=== FULL DICOM HEADER ===")
print(dcm)

Reading: ../data/dice_files/1.3.6.1.4.1.55648.52489457771155006307501275967031550463/1.3.6.1.4.1.55648.52489457771155006307501275967031550463.2/1.3.6.1.4.1.55648.52489457771155006307501275967031550463.2.2.green.dcm

=== FULL DICOM HEADER ===
Dataset.file_meta -------------------------------
(0002,0000) File Meta Information Group Length  UL: 224
(0002,0001) File Meta Information Version       OB: b'0 1\x00'
(0002,0002) Media Storage SOP Class UID         UI: MR Image Storage
(0002,0003) Media Storage SOP Instance UID      UI: 1.3.6.1.4.1.55648.52489457771155006307501275967031550463.2.2
(0002,0010) Transfer Syntax UID                 UI: JPEG Lossless, Non-Hierarchical, First-Order Prediction (Process 14 [Selection Value 1])
(0002,0012) Implementation Class UID            UI: 1.3.6.1.4.1.55648.0.1.999.0
(0002,0013) Implementation Version Name         SH: 'Segmed_v1.999.0 profile: default'
-------------------------------------------------
(0008,0005) Specific Character Set              C

/Users/shriyanshraj/miniforge3/envs/vlm_eval/lib/python3.10/site-packages/pydicom/valuerep.py:440: UserWarning: The value length (32) exceeds the maximum length of 16 allowed for VR SH.
  warn_and_log(msg)


In [23]:
pip install pylibjpeg pylibjpeg-libjpeg

  Using cached numpy-2.2.6-cp310-cp310-macosx_14_0_arm64.whl.metadata (62 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 600.3/600.3 kB 2.3 MB/s  0:00:0036m-:--:--
Using cached numpy-2.2.6-cp310-cp310-macosx_14_0_arm64.whl (5.3 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pylibjpeg]/3 [pylibjpeg]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikit-learn 1.3.2 requires numpy<2.0,>=1.17.3, but you have numpy 2.2.6 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [10]:
import pydicom
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import os

def apply_windowing(pixel_array: np.ndarray, window_center: float, window_width: float) -> np.ndarray:
    """
    Apply DICOM windowing to convert raw pixel values to display values.
    This is the correct way to render MRI/CT images for human (and model) viewing.
    Without this, images appear washed out or too dark.
    """
    low  = window_center - window_width / 2
    high = window_center + window_width / 2
    
    arr = pixel_array.astype(np.float32)
    arr = np.clip(arr, low, high)
    arr = (arr - low) / (high - low) * 255.0
    return arr.astype(np.uint8)

def dicom_to_pil(dcm_path: str) -> Image.Image:
    """Convert a DICOM file to a PIL RGB image using embedded window values."""
    dcm   = pydicom.dcmread(dcm_path)
    array = dcm.pixel_array
    
    # Use embedded window values if available, else fall back to min-max
    wc = float(getattr(dcm, 'WindowCenter', None) or (array.max() + array.min()) / 2)
    ww = float(getattr(dcm, 'WindowWidth',  None) or (array.max() - array.min()))
    
    # Handle case where WC/WW are lists (some DICOM files have multiple windows)
    if isinstance(wc, (list, pydicom.multival.MultiValue)): wc = float(wc[0])
    if isinstance(ww, (list, pydicom.multival.MultiValue)): ww = float(ww[0])
    
    windowed = apply_windowing(array, wc, ww)
    return Image.fromarray(windowed).convert('RGB')

# Test on all series — visualise one slice from each
dataset_path = "../data/dice_files/"
series_samples = {}

for root, dirs, files in os.walk(dataset_path):
    dcm_files = sorted([f for f in files if f.endswith('.dcm')])
    if dcm_files:
        # Take middle slice of each series for representative view
        mid_file = dcm_files[len(dcm_files) // 2]
        series_id = os.path.basename(root).split('.')[-1]
        series_samples[series_id] = os.path.join(root, mid_file)

fig, axes = plt.subplots(1, len(series_samples), figsize=(4 * len(series_samples), 4))
if len(series_samples) == 1:
    axes = [axes]

for ax, (sid, path) in zip(axes, sorted(series_samples.items())):
    dcm = pydicom.dcmread(path)
    img = dicom_to_pil(path)
    series_desc = getattr(dcm, 'SeriesDescription', f'Series {sid}')
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Series {sid}\n{series_desc}', fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig('dicom_all_series.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved dicom_all_series.png")

# Print series descriptions
print("\nSeries summary:")
for sid, path in sorted(series_samples.items()):
    dcm = pydicom.dcmread(path)
    n_files = len([f for f in os.listdir(os.path.dirname(path)) if f.endswith('.dcm')])
    print(f"  Series {sid}: {getattr(dcm, 'SeriesDescription', 'N/A')} — {n_files} slices")

Saved dicom_all_series.png

Series summary:
  Series 1: localizer_tra — 3 slices
  Series 2: localizer_sag+cor+tra — 6 slices
  Series 3: AX PD FS — 30 slices
  Series 4: SAG PD — 34 slices
  Series 5: SAG PD FS — 34 slices
  Series 6: COR PD FS — 24 slices
  Series 7: SAG PD FS — 26 slices
  Series 8: SAG PD THIN ACL — 14 slices


In [3]:
import io
import json
import os
import pandas as pd

# 1. Paste the raw Slack text into a multi-line Python string
raw_slack_data = """Study ID,Patient ID,Report,Patient Age,Patient Sex,Modality,Body Part,Study Date,Manufacturer,Model Name,Site ID,Source Location,Status,Metadata On Study Level,Metadata On Series Level,Zip File Index
1.3.6.1.4.1.55648.52489457771155006307501275967031550463,Segmed_Patient_6523170992291531753,"MAGNETIC RESONANCE INAGING LOWER EXTREMITY ANY JOINT W/O*
L/R: LEFT 

CLINICAL HISTORY:MEDIAL MENISCUS TEAR
Examination: LOWER EXTREMITY ANY JOINT W/O*
 
Clinical history: 42 years  old Female with MEDIAL MENISCUS TEAR
 
Comparison: None
 
Findings:
 
Multiplanar MRI of the left knee is performed.
 
Small effusion is seen. No chondromalacia patellar or Baker's cyst is
evident.
 
Anterior and posterior horns of both the medial and lateral meniscus
are intact without evidence of tear.
 
No bone bruise or fracture is seen.
 
Anterior cruciate ligament, posterior cruciate ligament, medial
collateral ligament, patellar ligament, and visualized lateral
collateral ligaments are intact.
 
*** IMPRESSION ***:
 
Small effusion
REPORT STATUS: Signed       <Electronically signed by segmed_DOCTOR in segmed_ADDRESSES> segmed_DATE_TIME
			segmed_DOCTOR

",42,,MR,Lower-limb/leg,2014-12-09T00:00:00Z,SIEMENS,Espree,,us_southwest_2,AVAILABLE,"{""Modality"": ""MR"", ""PatientID"": ""Segmed_Patient_6523170992291531753"", ""StudyDate"": ""20141209"", ""StudyTime"": ""090000.000000"", ""PatientAge"": ""042Y"", ""PatientSex"": ""F"", ""PatientName"": ""Segmed_Patient_6523170992291531753"", ""PatientSize"": ""1.5494031008333"", ""Manufacturer"": ""SIEMENS"", ""ProtocolName"": ""SAG PD THIN ACL"", ""PatientWeight"": ""90.71848554"", ""SoftwareVersions"": ""syngo MR B19"", ""StudyInstanceUID"": ""1.3.6.1.4.1.55648.52489457771155006307501275967031550463"", ""TransferSyntaxUID"": ""1.2.840.10008.1.2.4.70"", ""ManufacturerModelName"": ""Espree"", ""ImplementationClassUID"": ""1.3.6.1.4.1.55648.0.1.999.0"", ""ImplementationVersionName"": ""Segmed_v1.999.0 profile: default"", ""FileMetaInformationVersion"": ""48 32 49 0"", ""FileMetaInformationGroupLength"": ""224""}","{""1.3.6.1.4.1.55648.52489457771155006307501275967031550463.2"": {""SeriesNumber"": ""2"", ""PatientPosition"": ""FFS"", ""BodyPartExamined"": ""KNEE"", ""SeriesDescription"": ""localizer_sag+cor+tra"", ""SeriesInstanceUID"": ""1.3.6.1.4.1.55648.52489457771155006307501275967031550463.2""}, ""1.3.6.1.4.1.55648.52489457771155006307501275967031550463.3"": {""SeriesNumber"": ""3"", ""PatientPosition"": ""FFS"", ""BodyPartExamined"": ""KNEE"", ""SeriesDescription"": ""AXIAL PD FS"", ""SeriesInstanceUID"": ""1.3.6.1.4.1.55648.52489457771155006307501275967031550463.3""}, ""1.3.6.1.4.1.55648.52489457771155006307501275967031550463.4"": {""SeriesNumber"": ""4"", ""PatientPosition"": ""FFS"", ""BodyPartExamined"": ""KNEE"", ""SeriesDescription"": ""COR PD FS"", ""SeriesInstanceUID"": ""1.3.6.1.4.1.55648.52489457771155006307501275967031550463.4""}, ""1.3.6.1.4.1.55648.52489457771155006307501275967031550463.5"": {""SeriesNumber"": ""5"", ""PatientPosition"": ""FFS"", ""BodyPartExamined"": ""KNEE"", ""SeriesDescription"": ""COR T1"", ""SeriesInstanceUID"": ""1.3.6.1.4.1.55648.52489457771155006307501275967031550463.5""}, ""1.3.6.1.4.1.55648.52489457771155006307501275967031550463.6"": {""SeriesNumber"": ""6"", ""PatientPosition"": ""FFS"", ""BodyPartExamined"": ""KNEE"", ""SeriesDescription"": ""SAG PD"", ""SeriesInstanceUID"": ""1.3.6.1.4.1.55648.52489457771155006307501275967031550463.6""}, ""1.3.6.1.4.1.55648.52489457771155006307501275967031550463.7"": {""SeriesNumber"": ""7"", ""PatientPosition"": ""FFS"", ""BodyPartExamined"": ""KNEE"", ""SeriesDescription"": ""SAG PD FS"", ""SeriesInstanceUID"": ""1.3.6.1.4.1.55648.52489457771155006307501275967031550463.7""}, ""1.3.6.1.4.1.55648.52489457771155006307501275967031550463.8"": {""SeriesNumber"": ""8"", ""PatientPosition"": ""FFS"", ""BodyPartExamined"": ""KNEE"", ""SeriesDescription"": ""SAG PD THIN ACL"", ""SeriesInstanceUID"": ""1.3.6.1.4.1.55648.52489457771155006307501275967031550463.8""}}",0
1.3.6.1.4.1.55648.6974691373277018395743151067111743,Segmed_Patient_3126784625671810643,"MRI-LEFT KNEE NON CONTRAST HISTORY: M25.562 Left knee pain TECHNIQUE: Left knee imaged on a 1.5 Tesla high-field wide-bore MR scanner using multiplanar multisequence technique. COMPARISON: No prior studies are available for direct comparison at the time of this interpretation. FINDINGS: Ligaments: The anterior cruciate, posterior cruciate, medial collateral, and lateral collateral ligaments are intact. Menisci: There is no evidence for medial or lateral meniscal tears. Extensor Mechanism: The quadriceps and patellar tendons are intact. The medial and lateral patellofemoral ligaments are unremarkable. Effusion: No significant amount of knee joint fluid is present. Soft tissue edema is present on the anterior aspect of the knee. Cartilage: Focal cartilage loss involving more than 50% the cartilage thickness is present on the median ridge of the patella. Less severe cartilage loss is present femorotibial compartments. Bone Marrow: Overall, the bone marrow signal is age-appropriate. Popliteal fossa: There is no significant popliteal cyst. A small amount of fluid is present in the popliteus tendon sheath. Iliotibial band: A small amount of fluid is present in the popliteus tendon sheath. Posterolateral corner: The posterior lateral corner is unremarkable. ----- Page Break ----- ------------------------- IMPRESSION: Cartilage loss most pronounced on the patella. M17.12 Soft tissue edema on the anterior aspect of the knee. R60.0 No evidence for meniscal or ligament tears. Signed by: segmed_FIRSTNAME segmed_LASTNAME Signed Date: MM/DD/YYYY TIMESTAMP ",67,F,MR,Knee,2018-07-29T00:00:00Z,SIEMENS,Aera,segmed_76,us_east,AVAILABLE,"{""Modality"": ""MR"", ""PatientID"": ""Segmed_Patient_3126784625671810643"", ""StudyDate"": ""20180729"", ""PatientAge"": ""067Y"", ""PatientSex"": ""F"", ""PatientName"": ""Segmed_Patient_3126784625671810643"", ""PatientSize"": ""1.6510033041667"", ""Manufacturer"": ""SIEMENS"", ""ProtocolName"": ""COR PD FS"", ""PatientWeight"": ""75.7499354259"", ""SoftwareVersions"": ""syngo MR E11"", ""StudyInstanceUID"": ""1.3.6.1.4.1.55648.6974691373277018395743151067111743"", ""TransferSyntaxUID"": ""1.2.840.10008.1.2.4.70"", ""ManufacturerModelName"": ""Aera"", ""ImplementationClassUID"": ""1.3.6.1.4.1.55648.0.v1.982.2"", ""ImplementationVersionName"": ""Segmed_vv1.982.2 profile: default"", ""FileMetaInformationVersion"": ""0 1"", ""SourceApplicationEntityTitle"": ""Segmed.ai"", ""FileMetaInformationGroupLength"": ""238""}","{""1.3.6.1.4.1.55648.6974691373277018395743151067111743.1"": {""SeriesNumber"": ""1"", ""PatientPosition"": ""FFS"", ""BodyPartExamined"": ""EXTREMITY"", ""SeriesDescription"": ""localizer_tra"", ""SeriesInstanceUID"": ""1.3.6.1.4.1.55648.6974691373277018395743151067111743.1""}, ""1.3.6.1.4.1.55648.6974691373277018395743151067111743.2"": {""SeriesNumber"": ""2"", ""PatientPosition"": ""FFS"", ""BodyPartExamined"": ""EXTREMITY"", ""SeriesDescription"": ""localizer_sag+cor+tra"", ""SeriesInstanceUID"": ""1.3.6.1.4.1.55648.6974691373277018395743151067111743.2""}, ""1.3.6.1.4.1.55648.6974691373277018395743151067111743.3"": {""SeriesNumber"": ""3"", ""PatientPosition"": ""FFS"", ""BodyPartExamined"": ""EXTREMITY"", ""SeriesDescription"": ""AX PD FS"", ""SeriesInstanceUID"": ""1.3.6.1.4.1.55648.6974691373277018395743151067111743.3""}, ""1.3.6.1.4.1.55648.6974691373277018395743151067111743.4"": {""SeriesNumber"": ""4"", ""PatientPosition"": ""FFS"", ""BodyPartExamined"": ""EXTREMITY"", ""SeriesDescription"": ""SAG PD"", ""SeriesInstanceUID"": ""1.3.6.1.4.1.55648.6974691373277018395743151067111743.4""}, ""1.3.6.1.4.1.55648.6974691373277018395743151067111743.5"": {""SeriesNumber"": ""5"", ""PatientPosition"": ""FFS"", ""BodyPartExamined"": ""EXTREMITY"", ""SeriesDescription"": ""SAG PD FS"", ""SeriesInstanceUID"": ""1.3.6.1.4.1.55648.6974691373277018395743151067111743.5""}, ""1.3.6.1.4.1.55648.6974691373277018395743151067111743.6"": {""SeriesNumber"": ""6"", ""PatientPosition"": ""FFS"", ""BodyPartExamined"": ""EXTREMITY"", ""SeriesDescription"": ""COR PD FS"", ""SeriesInstanceUID"": ""1.3.6.1.4.1.55648.6974691373277018395743151067111743.6""}}",0
"""

# 2. Save it locally as a clean CSV file
csv_path = "../data/clinical_metadata.csv"
os.makedirs(os.path.dirname(csv_path), exist_ok=True)
with open(csv_path, "w", encoding="utf-8") as f:
    f.write(raw_slack_data.strip())

# 3. Load with pandas to verify structure
df = pd.read_csv(csv_path)
print(f"Successfully saved and parsed dataset structure. Found {len(df)} patient records.")

# 4. Helper Function to look up report string by any local file path
def get_report_for_file(file_path):
    # Extract the study UID from path (e.g., '1.3.6...550463')
    parts = file_path.split(os.sep)
    study_id = next((p for p in parts if p.startswith("1.3.6.")), None)
    
    if not study_id:
        return "Could not resolve Study ID from file path."
    
    # Strip any subfolder suffixes to get base study UID
    base_study_id = study_id.split('.')[0] if '.' in study_id and not study_id.startswith('1.3.6') else study_id
    # Ensure exact lookup match for top-level folder names
    match = df[df['Study ID'].str.strip() == study_id.strip()]
    if not match.empty:
        return match.iloc[0]['Report']
    return f"No matching report found for Study ID: {study_id}"

# Quick Test lookup
sample_file = "../data/dice_files/1.3.6.1.4.1.55648.52489457771155006307501275967031550463/1.3.6.1.4.1.55648.52489457771155006307501275967031550463.3/sample.dcm"
print("\n=== Sample Report Lookup Test ===")
print(get_report_for_file(sample_file)[:300] + "...")

Successfully saved and parsed dataset structure. Found 2 patient records.

=== Sample Report Lookup Test ===
MAGNETIC RESONANCE INAGING LOWER EXTREMITY ANY JOINT W/O*
L/R: LEFT 

CLINICAL HISTORY:MEDIAL MENISCUS TEAR
Examination: LOWER EXTREMITY ANY JOINT W/O*
 
Clinical history: 42 years  old Female with MEDIAL MENISCUS TEAR
 
Comparison: None
 
Findings:
 
Multiplanar MRI of the left knee is performed.
 ...


In [2]:
import json
import os

# Define the 16 Ground Truth QA Pairs
qa_dataset = [
    # ================= PATIENT 1 =================
    {
        "study_id": "1.3.6.1.4.1.55648.52489457771155006307501275967031550463",
        "target_series": "6",
        "question": "Is there evidence of a medial meniscus tear in this scan?",
        "answer": "No",
        "answer_type": "CLOSED"
    },
    {
        "study_id": "1.3.6.1.4.1.55648.52489457771155006307501275967031550463",
        "target_series": "8",
        "question": "Are the anterior and posterior cruciate ligaments intact?",
        "answer": "Yes",
        "answer_type": "CLOSED"
    },
    {
        "study_id": "1.3.6.1.4.1.55648.52489457771155006307501275967031550463",
        "target_series": "7",
        "question": "What is the primary abnormal fluid finding in this joint?",
        "answer": "Small effusion",
        "answer_type": "OPEN"
    },
    {
        "study_id": "1.3.6.1.4.1.55648.52489457771155006307501275967031550463",
        "target_series": "3",
        "question": "Is a Baker's cyst evident in the popliteal region?",
        "answer": "No",
        "answer_type": "CLOSED"
    },
    {
        "study_id": "1.3.6.1.4.1.55648.52489457771155006307501275967031550463",
        "target_series": "4",
        "question": "What is the condition of the medial and lateral collateral ligaments?",
        "answer": "Intact",
        "answer_type": "OPEN"
    },
    {
        "study_id": "1.3.6.1.4.1.55648.52489457771155006307501275967031550463",
        "target_series": "7",
        "question": "Is there any evidence of a bone bruise or fracture?",
        "answer": "No",
        "answer_type": "CLOSED"
    },
    {
        "study_id": "1.3.6.1.4.1.55648.52489457771155006307501275967031550463",
        "target_series": "3",
        "question": "Is chondromalacia patellar present in this patient?",
        "answer": "No",
        "answer_type": "CLOSED"
    },
    {
        "study_id": "1.3.6.1.4.1.55648.52489457771155006307501275967031550463",
        "target_series": "7",
        "question": "What is the overall impression of this MRI?",
        "answer": "Small effusion",
        "answer_type": "OPEN"
    },

    # ================= PATIENT 2 =================
    {
        "study_id": "1.3.6.1.4.1.55648.6974691373277018395743151067111743",
        "target_series": "5",
        "question": "Where is the most pronounced focal cartilage loss located?",
        "answer": "Median ridge of the patella",
        "answer_type": "OPEN"
    },
    {
        "study_id": "1.3.6.1.4.1.55648.6974691373277018395743151067111743",
        "target_series": "4",
        "question": "Are there any tears present in the medial or lateral meniscus?",
        "answer": "No",
        "answer_type": "CLOSED"
    },
    {
        "study_id": "1.3.6.1.4.1.55648.6974691373277018395743151067111743",
        "target_series": "5",
        "question": "What finding is noted on the anterior aspect of the knee?",
        "answer": "Soft tissue edema",
        "answer_type": "OPEN"
    },
    {
        "study_id": "1.3.6.1.4.1.55648.6974691373277018395743151067111743",
        "target_series": "5",
        "question": "Is there a significant amount of knee joint fluid (effusion) present?",
        "answer": "No",
        "answer_type": "CLOSED"
    },
    {
        "study_id": "1.3.6.1.4.1.55648.6974691373277018395743151067111743",
        "target_series": "5",
        "question": "Are the quadriceps and patellar tendons intact?",
        "answer": "Yes",
        "answer_type": "CLOSED"
    },
    {
        "study_id": "1.3.6.1.4.1.55648.6974691373277018395743151067111743",
        "target_series": "5",
        "question": "What is the severity of the cartilage loss on the patella?",
        "answer": "More than 50% thickness",
        "answer_type": "OPEN"
    },
    {
        "study_id": "1.3.6.1.4.1.55648.6974691373277018395743151067111743",
        "target_series": "3",
        "question": "Is there a significant popliteal cyst present?",
        "answer": "No",
        "answer_type": "CLOSED"
    },
    {
        "study_id": "1.3.6.1.4.1.55648.6974691373277018395743151067111743",
        "target_series": "6",
        "question": "What finding is present in the popliteus tendon sheath?",
        "answer": "Small amount of fluid",
        "answer_type": "OPEN"
    }
]

# Write to file
output_path = "../data/clinical_vqa_dataset.jsonl"
os.makedirs(os.path.dirname(output_path), exist_ok=True)

with open(output_path, 'w') as f:
    for item in qa_dataset:
        f.write(json.dumps(item) + '\n')

print(f"Successfully generated {len(qa_dataset)} QA pairs at: {output_path}")

Successfully generated 16 QA pairs at: ../data/clinical_vqa_dataset.jsonl


In [3]:
import subprocess
subprocess.run(['pip', 'install', 'pydicom', 'pylibjpeg', 'pylibjpeg-libjpeg', '-q'])
print('Done.')

Done.


In [4]:
import os, json, re, string
import torch
import numpy as np
import pydicom
import pydicom.multival
from PIL import Image
from collections import Counter
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from sacrebleu.metrics import BLEU
from tqdm import tqdm
import nltk

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

# Adjusted to ensure CUDA is picked up on Kaggle T4 GPUs
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject